# 16. Quality Profiles and Rubrics — Configure runtime quality gates

Agents need more than a final answer; they need a definition of acceptable work. This chapter separates runtime profiles from rubrics and shows how to turn quality expectations into executable checks.

**Learning goals**
- Define rubrics as completion criteria.
- Use deterministic evaluators for simple quality gates.
- Distinguish runtime profiles from answer-quality rubrics.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# Observability setup — disabled when keys are not present.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")
lf_config = {}

In [ ]:
from deepagents import HarnessProfile, register_harness_profile

profile = HarnessProfile(system_prompt_suffix="Respond with concise Korean bullets.")
register_harness_profile("course:demo", profile)

type(profile).__name__

## 16.1 Rubrics define completion criteria

A rubric translates “good enough” into observable criteria. It helps agents and reviewers agree on what must be present before work is considered complete.


In [ ]:
rubric = [
    {"name": "has_summary", "check": lambda text: "summary" in text},
    {"name": "has_next_step", "check": lambda text: "next" in text},
]

rubric[0]["name"]

## 16.2 Deterministic rubric evaluator

`RubricMiddleware` requires criterion-level evaluation coverage and connects to grader hooks and a runtime self-revision loop. The evaluator in this cell is a deterministic teaching miniature, not the SDK middleware itself.

That boundary matters: the teaching evaluator shows the contract without an LLM call, while `RubricMiddleware` checks output quality during runtime.

In [ ]:
def evaluate(text: str) -> dict:
    results = {item["name"]: item["check"](text) for item in rubric}
    return {"passed": all(results.values()), "criteria": results}

evaluate("summary: done. next: run tests")

## 16.3 Profile versus rubric

| Item | Role |
|---|---|
| profile | Tunes model/provider harness defaults, tool descriptions, excluded tools, and extra middleware |
| rubric | Defines quality criteria the final result must satisfy |
| eval | Runs batch/offline regression checks |
| middleware | Runs runtime self-revision and grader hooks |

A profile changes the runtime environment; a rubric changes the completion contract. Keeping them separate makes regressions easier to diagnose.

---

## Summary

| Item | Content |
|---|---|
| **Covered** | `HarnessProfile`, rubric criteria, deterministic evaluator, and the `RubricMiddleware` contract |
| **Core idea** | Profiles configure runtime behavior; rubrics define completion criteria. |
| **Caution** | Do not describe a deterministic teaching evaluator as if it were SDK `RubricMiddleware`. |

## Reference docs

- [`profiles.md`](../../docs/deepagents/profiles.md)
- [`rubric.md`](../../docs/deepagents/rubric.md)
- [`going-to-production.md`](../../docs/deepagents/going-to-production.md)